In [7]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, Input, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

## Étape 1 — Vérification de l’environnement et des données

Avant de commencer l’analyse, nous vérifions la version de Python, la disponibilité des bibliothèques nécessaires et la présence du fichier de données avec les colonnes attendues.

**Objectif :** repérer dès le départ les problèmes d’environnement ou de format qui pourraient interrompre les étapes suivantes.

In [8]:
print(f"Version de Python : {sys.version.split()[0]}")
if sys.version_info > (3, 13, 5):
    print("Attention : les consignes recommandent Python 3.13.5 ou une version antérieure pour Keras/TensorFlow.")

packages = ["pandas", "numpy", "scikit-learn", "nltk", "tensorflow", "torch", "gensim"]
for package in packages:
    try:
        print(f"{package} : {version(package)}")
    except PackageNotFoundError:
        print(f"{package} : non installé")

DATA_PATH = Path("bbc-news-data.csv")
if not DATA_PATH.is_file():
    DATA_PATH = Path("Apply SVM to predict BBC news category") / "bbc-news-data.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Fichier de données introuvable : {DATA_PATH.resolve()}")

expected_columns = ["category", "filename", "title", "content"]
actual_columns = pd.read_csv(DATA_PATH, sep="\t", nrows=0).columns.tolist()
print(f"\nFichier de données : {DATA_PATH.resolve()}")
print(f"Colonnes détectées : {actual_columns}")
missing_columns = [column for column in expected_columns if column not in actual_columns]
if missing_columns:
    raise ValueError(f"Colonnes attendues manquantes : {missing_columns}")
print("Vérification du fichier réussie : toutes les colonnes attendues sont présentes.")

Version de Python : 3.13.14
Attention : les consignes recommandent Python 3.13.5 ou une version antérieure pour Keras/TensorFlow.
pandas : 3.0.6
numpy : 2.5.3
scikit-learn : 1.9.1
nltk : 3.10.3
tensorflow : 2.21.0
torch : 2.14.1
gensim : 4.4.0

Fichier de données : C:\Users\boniv\Downloads\Github Project\ML-Trainning-and-evaluation\Apply SVM to predict BBC news category\bbc-news-data.csv
Colonnes détectées : ['category', 'filename', 'title', 'content']
Vérification du fichier réussie : toutes les colonnes attendues sont présentes.


## Étape 2 — Chargement et exploration du jeu de données

Nous chargeons le fichier tabulé dans un DataFrame, puis observons sa taille, les premières lignes, la répartition des catégories, les valeurs manquantes et les lignes dupliquées. La colonne `filename` sert uniquement d’identifiant et ne sera pas utilisée comme texte d’entrée du modèle.

**Objectif :** comprendre la structure et la qualité des données afin de savoir si un nettoyage sera nécessaire avant l’entraînement.

In [9]:
df = pd.read_csv(DATA_PATH, sep="\t")

print(f"Dimensions du jeu de données : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nTypes des colonnes :\n{df.dtypes}")
print("\nValeurs manquantes par colonne :")
display(df.isna().sum().to_frame(name="valeurs_manquantes"))

print(f"\nNombre de lignes dupliquées : {df.duplicated().sum()}")
print(f"\nNombre de catégories : {df['category'].nunique()}")
print("Répartition des catégories :")
display(df["category"].value_counts().rename_axis("category").to_frame(name="nombre_articles"))

print("\nAperçu de quelques articles :")
display(df[["category", "filename", "title", "content"]].head())

Dimensions du jeu de données : 2225 lignes × 4 colonnes

Types des colonnes :
category    str
filename    str
title       str
content     str
dtype: object

Valeurs manquantes par colonne :


,valeurs_manquantes
category,0
filename,0
title,0
content,0



Nombre de lignes dupliquées : 0

Nombre de catégories : 5
Répartition des catégories :


,nombre_articles
category,
sport,511
business,510
politics,417
tech,401
entertainment,386



Aperçu de quelques articles :


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


## Étape 3 — Prétraitement du texte avec NLTK

Nous réunissons le titre et le contenu dans `raw_text`, puis mettons le texte en minuscules, le tokenisons, conservons uniquement les mots composés de lettres, retirons les mots vides anglais et lemmatisons les tokens. Nous gardons les tokens nettoyés dans `tokens` et leur version réunie dans `processed_text` pour les vectoriseurs classiques.

**Objectif :** normaliser les articles pour réduire le bruit textuel et préparer des représentations adaptées aux différents modèles.

In [10]:
nltk_resources = {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab",
    "stopwords": "corpora/stopwords",
    "wordnet": "corpora/wordnet",
    "omw-1.4": "corpora/omw-1.4",
}
for resource_name, resource_path in nltk_resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        if not nltk.download(resource_name, quiet=True):
            raise RuntimeError(f"Impossible de télécharger la ressource NLTK : {resource_name}")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    alphabetic_tokens = [token for token in tokens if token.isalpha()]
    return [
        lemmatizer.lemmatize(token)
        for token in alphabetic_tokens
        if token not in stop_words
    ]

df["raw_text"] = (
    df["title"].fillna("").astype(str).str.strip()
    + " "
    + df["content"].fillna("").astype(str).str.strip()
).str.strip()
df["tokens"] = df["raw_text"].apply(preprocess_text)
df["processed_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

display(df[["category", "raw_text", "processed_text"]].head(3))
print(f"Articles prétraités : {len(df)}")
print(f"Articles sans token après nettoyage : {(df['tokens'].str.len() == 0).sum()}")

,category,raw_text,processed_text
0,business,Ad sales boost Time Warner profit Quarterly pr...,ad sale boost time warner profit quarterly pro...
1,business,Dollar gains on Greenspan speech The dollar ha...,dollar gain greenspan speech dollar hit highes...
2,business,Yukos unit buyer faces loan claim The owners o...,yukos unit buyer face loan claim owner embattl...


Articles prétraités : 2225
Articles sans token après nettoyage : 0


## Étape 4 — Séparation des données en entraînement et test

Nous réservons 20 % des articles pour le test et utilisons une séparation stratifiée, afin de conserver la proportion de chaque catégorie dans les deux ensembles. La graine aléatoire fixe rend la séparation reproductible. Les mêmes indices sont appliqués au texte nettoyé, aux tokens et aux étiquettes.

**Objectif :** préparer une évaluation équitable sur des articles qui ne seront pas utilisés pendant l’entraînement. Les vectoriseurs devront ensuite être ajustés uniquement sur le texte d’entraînement.

In [11]:
SEED = 42
TEST_SIZE = 0.20
article_indices = np.arange(len(df))

train_indices, test_indices = train_test_split(
    article_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["category"],
)

X_train_text = df["processed_text"].iloc[train_indices].reset_index(drop=True)
X_test_text = df["processed_text"].iloc[test_indices].reset_index(drop=True)
X_train_tokens = df["tokens"].iloc[train_indices].tolist()
X_test_tokens = df["tokens"].iloc[test_indices].tolist()
y_train = df["category"].iloc[train_indices].reset_index(drop=True)
y_test = df["category"].iloc[test_indices].reset_index(drop=True)

if set(train_indices).intersection(test_indices):
    raise RuntimeError("Les ensembles d'entraînement et de test se chevauchent.")
if len(train_indices) + len(test_indices) != len(df):
    raise RuntimeError("La séparation n'a pas conservé tous les articles.")

print(f"Articles d'entraînement : {len(X_train_text)} ({len(X_train_text) / len(df):.1%})")
print(f"Articles de test : {len(X_test_text)} ({len(X_test_text) / len(df):.1%})")
print("\nRépartition des catégories (pourcentage dans chaque ensemble) :")
class_distribution = pd.concat(
    [
        y_train.value_counts(normalize=True).mul(100).rename("train (%)"),
        y_test.value_counts(normalize=True).mul(100).rename("test (%)"),
    ],
    axis=1,
).sort_index()
display(class_distribution.round(2))

Articles d'entraînement : 1780 (80.0%)
Articles de test : 445 (20.0%)

Répartition des catégories (pourcentage dans chaque ensemble) :


,train (%),test (%)
category,,
business,22.92,22.92
entertainment,17.36,17.30
politics,18.71,18.88
sport,22.98,22.92
tech,18.03,17.98


## Étape 5 — Entraînement et comparaison des combinaisons

Nous testons les sept combinaisons demandées : CountVectorizer + régression logistique, TF-IDF + régression logistique, TF-IDF + MLP, TF-IDF + réseau dense Keras, CountVectorizer + réseau dense Keras, Word2Vec + BiLSTM et TF-IDF + LinearSVC. Une petite partie de l’ensemble d’entraînement sert de validation ; le jeu de test reste à part. Tous les vectoriseurs et Word2Vec sont ajustés uniquement sur les données servant à l’entraînement.

Pour chaque combinaison, nous calculons l’accuracy et le F1 macro sur validation et sur test. <br>
**Objectif :** comparer les performances de façon cohérente, retenir le modèle d’après le F1 macro de validation (qui traite chaque catégorie de manière égale), puis présenter ses résultats de test sans utiliser le test pour le choix du modèle. En cas d’égalité à la précision affichée, nous privilégions LinearSVC, adapté à la classification de texte avec des matrices creuses et directement lié au sujet SVM.

In [12]:
fit_positions, val_positions = train_test_split(
    np.arange(len(y_train)),
    test_size=0.10,
    random_state=SEED,
    stratify=y_train,
)
X_fit_text = X_train_text.iloc[fit_positions].reset_index(drop=True)
X_val_text = X_train_text.iloc[val_positions].reset_index(drop=True)
X_fit_tokens = [X_train_tokens[i] for i in fit_positions]
X_val_tokens = [X_train_tokens[i] for i in val_positions]
y_fit = y_train.iloc[fit_positions].reset_index(drop=True)
y_val = y_train.iloc[val_positions].reset_index(drop=True)

results = []
predictions_by_model = {}

def record_result(name, validation_predictions, test_predictions):
    results.append(
        {
            "combinaison": name,
            "validation_accuracy": accuracy_score(y_val, validation_predictions),
            "validation_f1_macro": f1_score(y_val, validation_predictions, average="macro"),
            "test_accuracy": accuracy_score(y_test, test_predictions),
            "test_f1_macro": f1_score(y_test, test_predictions, average="macro"),
        }
    )
    predictions_by_model[name] = {
        "validation": validation_predictions,
        "test": test_predictions,
    }

classical_models = {
    "CountVectorizer + LogisticRegression": Pipeline(
        [
            ("vectorizer", CountVectorizer(max_features=30000)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=SEED)),
        ]
    ),
    "TF-IDF + LogisticRegression": Pipeline(
        [
            ("vectorizer", TfidfVectorizer(max_features=30000, sublinear_tf=True)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=SEED)),
        ]
    ),
    "TF-IDF + MLPClassifier": Pipeline(
        [
            ("vectorizer", TfidfVectorizer(max_features=20000, sublinear_tf=True)),
            (
                "classifier",
                MLPClassifier(
                    hidden_layer_sizes=(100,),
                    max_iter=100,
                    early_stopping=True,
                    n_iter_no_change=8,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "TF-IDF + LinearSVC": Pipeline(
        [
            ("vectorizer", TfidfVectorizer(max_features=30000, sublinear_tf=True)),
            ("classifier", LinearSVC(random_state=SEED)),
        ]
    ),
}

for model_name, model in classical_models.items():
    model.fit(X_fit_text, y_fit)
    record_result(
        model_name,
        model.predict(X_val_text),
        model.predict(X_test_text),
    )
    print(f"Entraînement terminé : {model_name}")

label_encoder = LabelEncoder().fit(y_fit)
y_fit_encoded = label_encoder.transform(y_fit)
y_val_encoded = label_encoder.transform(y_val)
num_classes = len(label_encoder.classes_)

def build_dense_model(input_size):
    model = Sequential(
        [
            Input(shape=(input_size,)),
            Dense(128, activation="relu"),
            Dropout(0.35),
            Dense(64, activation="relu"),
            Dropout(0.25),
            Dense(num_classes, activation="softmax"),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def fit_dense_text_model(model_name, vectorizer):
    train_matrix = vectorizer.fit_transform(X_fit_text).toarray().astype(np.float32)
    val_matrix = vectorizer.transform(X_val_text).toarray().astype(np.float32)
    test_matrix = vectorizer.transform(X_test_text).toarray().astype(np.float32)

    tf.keras.utils.set_random_seed(SEED)
    model = build_dense_model(train_matrix.shape[1])
    model.fit(
        train_matrix,
        y_fit_encoded,
        validation_data=(val_matrix, y_val_encoded),
        epochs=20,
        batch_size=32,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=0,
    )
    validation_predictions = label_encoder.inverse_transform(
        np.argmax(model.predict(val_matrix, verbose=0), axis=1)
    )
    test_predictions = label_encoder.inverse_transform(
        np.argmax(model.predict(test_matrix, verbose=0), axis=1)
    )
    record_result(model_name, validation_predictions, test_predictions)
    print(f"Entraînement terminé : {model_name}")

fit_dense_text_model(
    "TF-IDF + réseau dense Keras",
    TfidfVectorizer(max_features=5000, sublinear_tf=True),
)
fit_dense_text_model(
    "CountVectorizer + réseau dense Keras",
    CountVectorizer(max_features=5000, binary=True),
)

MAX_WORDS = 20000
MAX_SEQUENCE_LENGTH = 300
EMBEDDING_DIM = 100
word2vec_model = Word2Vec(
    sentences=X_fit_tokens,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    seed=SEED,
    epochs=20,
)
text_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<UNK>", filters="", lower=False)
text_tokenizer.fit_on_texts(X_fit_tokens)
vocabulary_size = min(MAX_WORDS, len(text_tokenizer.word_index) + 1)
embedding_matrix = np.zeros((vocabulary_size, EMBEDDING_DIM), dtype=np.float32)
for word, word_index in text_tokenizer.word_index.items():
    if word_index < vocabulary_size and word in word2vec_model.wv:
        embedding_matrix[word_index] = word2vec_model.wv[word]

def make_padded_sequences(token_lists):
    sequences = text_tokenizer.texts_to_sequences(token_lists)
    return pad_sequences(
        sequences,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post",
    )

X_fit_sequences = make_padded_sequences(X_fit_tokens)
X_val_sequences = make_padded_sequences(X_val_tokens)
X_test_sequences = make_padded_sequences(X_test_tokens)
tf.keras.utils.set_random_seed(SEED)
bilstm_model = Sequential(
    [
        Embedding(
            input_dim=vocabulary_size,
            output_dim=EMBEDDING_DIM,
            weights=[embedding_matrix],
            mask_zero=True,
            trainable=True,
        ),
        Bidirectional(LSTM(64)),
        Dropout(0.35),
        Dense(num_classes, activation="softmax"),
    ]
)
bilstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
bilstm_model.fit(
    X_fit_sequences,
    y_fit_encoded,
    validation_data=(X_val_sequences, y_val_encoded),
    epochs=12,
    batch_size=32,
    callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
    verbose=0,
)
bilstm_val_predictions = label_encoder.inverse_transform(
    np.argmax(bilstm_model.predict(X_val_sequences, verbose=0), axis=1)
)
bilstm_test_predictions = label_encoder.inverse_transform(
    np.argmax(bilstm_model.predict(X_test_sequences, verbose=0), axis=1)
)
record_result("Word2Vec + BiLSTM", bilstm_val_predictions, bilstm_test_predictions)
print("Entraînement terminé : Word2Vec + BiLSTM")

results_df = pd.DataFrame(results).sort_values(
    ["validation_f1_macro", "validation_accuracy"],
    ascending=False,
).reset_index(drop=True)
display(
    results_df.style.format(
        {
            "validation_accuracy": "{:.3f}",
            "validation_f1_macro": "{:.3f}",
            "test_accuracy": "{:.3f}",
            "test_f1_macro": "{:.3f}",
        }
    )
)
print("\nScores détaillés (accuracy et F1 macro) :")
print(results_df.to_string(index=False, float_format=lambda score: f"{score:.3f}"))

reported_f1 = results_df["validation_f1_macro"].round(3)
reported_accuracy = results_df["validation_accuracy"].round(3)
best_reported_f1 = reported_f1.max()
finalists = results_df.loc[reported_f1 == best_reported_f1].copy()
best_reported_accuracy = reported_accuracy.loc[finalists.index].max()
finalists = finalists.loc[reported_accuracy.loc[finalists.index] == best_reported_accuracy]
preferred_tie_break = "TF-IDF + LinearSVC"
if preferred_tie_break in finalists["combinaison"].values:
    selected_model_name = preferred_tie_break
else:
    selected_model_name = finalists.iloc[0]["combinaison"]
selected_model_test_predictions = predictions_by_model[selected_model_name]["test"]
selected_result = results_df.loc[results_df["combinaison"] == selected_model_name].iloc[0]
print(
    f"\nMeilleur modèle parmi les sept combinaisons imposées : {selected_model_name}. "
    f"Il obtient un F1 macro de validation de {selected_result['validation_f1_macro']:.3f} "
    f"et une accuracy de validation de {selected_result['validation_accuracy']:.3f}. "
    "Le F1 macro donne le même poids à chaque catégorie. En cas d'égalité aux trois décimales affichées, "
    "LinearSVC est privilégié car il est adapté aux données textuelles creuses et correspond au sujet SVM. "
    "Le jeu de test n'a pas servi à choisir ce modèle parmi les combinaisons imposées."
)
print("\nRapport détaillé du modèle retenu sur le jeu de test :")
print(
    classification_report(
        y_test,
        selected_model_test_predictions,
        labels=label_encoder.classes_,
        zero_division=0,
    )
)

Entraînement terminé : CountVectorizer + LogisticRegression
Entraînement terminé : TF-IDF + LogisticRegression
Entraînement terminé : TF-IDF + MLPClassifier
Entraînement terminé : TF-IDF + LinearSVC
Entraînement terminé : TF-IDF + réseau dense Keras
Entraînement terminé : CountVectorizer + réseau dense Keras
Entraînement terminé : Word2Vec + BiLSTM


,combinaison,validation_accuracy,validation_f1_macro,test_accuracy,test_f1_macro
0,TF-IDF + réseau dense Keras,0.989,0.989,0.984,0.984
1,TF-IDF + LinearSVC,0.989,0.989,0.991,0.991
2,TF-IDF + LogisticRegression,0.978,0.978,0.982,0.982
3,CountVectorizer + réseau dense Keras,0.972,0.972,0.984,0.984
4,TF-IDF + MLPClassifier,0.966,0.967,0.984,0.984
5,Word2Vec + BiLSTM,0.961,0.960,0.980,0.980
6,CountVectorizer + LogisticRegression,0.961,0.960,0.982,0.982



Scores détaillés (accuracy et F1 macro) :
                         combinaison  validation_accuracy  validation_f1_macro  test_accuracy  test_f1_macro
         TF-IDF + réseau dense Keras                0.989                0.989          0.984          0.984
                  TF-IDF + LinearSVC                0.989                0.989          0.991          0.991
         TF-IDF + LogisticRegression                0.978                0.978          0.982          0.982
CountVectorizer + réseau dense Keras                0.972                0.972          0.984          0.984
              TF-IDF + MLPClassifier                0.966                0.967          0.984          0.984
                   Word2Vec + BiLSTM                0.961                0.960          0.980          0.980
CountVectorizer + LogisticRegression                0.961                0.960          0.982          0.982

Modèle retenu : TF-IDF + LinearSVC. Il obtient un F1 macro de validation de 0.989 et

## Étape 6 — Tester une combinaison supplémentaire

Comme proposition complémentaire, nous testons TF-IDF avec des unigrammes et des bigrammes (`ngram_range=(1, 2)`), suivi de LinearSVC. Les bigrammes permettent au modèle de considérer certaines expressions de deux mots, tandis que LinearSVC reste efficace avec les vecteurs TF-IDF creux.

**Objectif :** vérifier sur validation si l’ajout de ces expressions améliore la classification par rapport à TF-IDF + LinearSVC avec unigrammes seulement. Nous ajoutons ensuite cette proposition au tableau comparatif. Le choix final repose sur la validation, et non sur les scores de test.

In [13]:
bigram_svc = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(
                max_features=40000,
                ngram_range=(1, 2),
                sublinear_tf=True,
            ),
        ),
        ("classifier", LinearSVC(random_state=SEED)),
    ]
)
bigram_svc.fit(X_fit_text, y_fit)
bigram_validation_predictions = bigram_svc.predict(X_val_text)
bigram_test_predictions = bigram_svc.predict(X_test_text)
bigram_model_name = "TF-IDF (1–2 grammes) + LinearSVC"
record_result(
    bigram_model_name,
    bigram_validation_predictions,
    bigram_test_predictions,
)

results_df = pd.DataFrame(results).sort_values(
    ["validation_f1_macro", "validation_accuracy"],
    ascending=False,
).reset_index(drop=True)
print("Scores actualisés des huit combinaisons (accuracy et F1 macro) :")
print(results_df.to_string(index=False, float_format=lambda score: f"{score:.3f}"))

bigram_result = results_df.loc[results_df["combinaison"] == bigram_model_name].iloc[0]
unigram_result = results_df.loc[
    results_df["combinaison"] == "TF-IDF + LinearSVC"
].iloc[0]
print(
    f"\nRésultat de la proposition : F1 macro validation = "
    f"{bigram_result['validation_f1_macro']:.3f}, accuracy test = "
    f"{bigram_result['test_accuracy']:.3f}."
)
validation_difference = (
    bigram_result["validation_f1_macro"] - unigram_result["validation_f1_macro"]
)
if validation_difference > 0.001:
    print(
        f"Les bigrammes améliorent le F1 macro de validation de "
        f"{validation_difference:.3f} par rapport à la version avec unigrammes seuls."
    )
elif validation_difference < -0.001:
    print(
        f"Les bigrammes diminuent le F1 macro de validation de "
        f"{abs(validation_difference):.3f} par rapport à la version avec unigrammes seuls ; "
        "leur ajout n'apporte donc pas d'amélioration dans cette expérience."
    )
else:
    print(
        "Le F1 macro de validation est très proche de celui de la version avec unigrammes seuls ; "
        "les bigrammes n'apportent pas de gain mesurable à cette précision."
    )

reported_f1 = results_df["validation_f1_macro"].round(3)
reported_accuracy = results_df["validation_accuracy"].round(3)
finalists = results_df.loc[reported_f1 == reported_f1.max()].copy()
finalists = finalists.loc[
    reported_accuracy.loc[finalists.index] == reported_accuracy.loc[finalists.index].max()
]
preferred_order = ["TF-IDF + LinearSVC", bigram_model_name]
selected_model_name = next(
    (name for name in preferred_order if name in finalists["combinaison"].values),
    finalists.iloc[0]["combinaison"],
)
selected_model_test_predictions = predictions_by_model[selected_model_name]["test"]
selected_result = results_df.loc[
    results_df["combinaison"] == selected_model_name
].iloc[0]
print(
    f"\nModèle retenu après comparaison des huit combinaisons : {selected_model_name}. "
    f"F1 macro validation = {selected_result['validation_f1_macro']:.3f}, "
    f"accuracy test = {selected_result['test_accuracy']:.3f}. "
    "En cas d'égalité aux trois décimales affichées, la version unigramme, plus simple, est préférée. "
    "Le test n'est pas utilisé pour sélectionner le modèle."
)
print("\nRapport du modèle final sur le jeu de test :")
print(
    classification_report(
        y_test,
        selected_model_test_predictions,
        labels=label_encoder.classes_,
        zero_division=0,
    )
)

Scores actualisés des huit combinaisons (accuracy et F1 macro) :
                         combinaison  validation_accuracy  validation_f1_macro  test_accuracy  test_f1_macro
         TF-IDF + réseau dense Keras                0.989                0.989          0.984          0.984
                  TF-IDF + LinearSVC                0.989                0.989          0.991          0.991
         TF-IDF + LogisticRegression                0.978                0.978          0.982          0.982
    TF-IDF (1–2 grammes) + LinearSVC                0.978                0.978          0.991          0.991
CountVectorizer + réseau dense Keras                0.972                0.972          0.984          0.984
              TF-IDF + MLPClassifier                0.966                0.967          0.984          0.984
                   Word2Vec + BiLSTM                0.961                0.960          0.980          0.980
CountVectorizer + LogisticRegression                0.961      